In [1]:
%load_ext autoreload
%autoreload 2

# --- Standard libs ---
import numpy as np
import pandas as pd
import time
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# --- Project imports ---
from Core.logging import flow_logger as logger

from Instruments.GX_271.gilson_ethernet import GilsonEthernet
from Instruments.GX_271.tray import Tray
from Instruments.GX_271.rack import Rack_209, Rack_3dp
from Instruments.GX_271.dim import DIM

from Instruments.WPI_Syr_pump.Pump import AL1000
from Instruments.Runze_valves.Runze62Valve import Runze62Valve
from Instruments.Knauer.knauer_pump import KnauerPump  # if applicable

from Ecosystems.reaction_segment_generation import RSG
from Ecosystems.segmentation import Segmentation

In [2]:
# --- Syringe pump (RSG) ---
syr_pump = AL1000("COM2")
syr_pump.connect()

# --- DIM ---
dim = DIM("COM5")
dim.connect()

# --- Runze valve ---
runze = Runze62Valve("COM8")
runze.connect()

# --- Carrier pump (Knauer) ---
k_pump = KnauerPump("COM4")
k_pump.connect()

AL1000 pump connected on COM2
Connected to Vici valve on COM5
Connected to Runze valve on COM8
[Runze] Valve at pos None -> state = UNKNOWN
[Runze] valve moved to pos 1 -> state = GAS_TO_DIM
Connected to Knauer pump on COM4


In [3]:
#--- Construct the tray ---
tray = Tray()

# --- Connect to Gilson ---
g = GilsonEthernet("192.168.1.101", admin_port=50185, tray=tray)

# --- Tell gilson instance about the DIM ---
g.dim = dim

# --- Start logging (declare this run belongs to the experiment "xxxxx") ---
logger.start_experiment("Segmentation_testing")

# --- Instantiate modules (racks, wash stations, etc.) (these know internal geometry) ---
rack1 = Rack_209()  
rack2 = Rack_3dp()

# --- Register modules on the tray with global offsets, previously handled by each module in the tray ---
tray.add_module(
    slot=1,
    name="rack1",
    module=rack1,
    x_offset=157,
    y_offset=9,
    x_min=145,
    x_max=250,
    y_min=2,
    y_max=292
)

tray.add_module(
    slot=2,
    name="rack2",
    module=rack2,
    x_offset=319,
    y_offset=39,
    x_min=275,
    x_max=370,
    y_min=2,
    y_max=292
) 

tray.add_module(
    slot=3,
    name="dim",
    module=dim,
    x_offset=9,     #These values are to be changed
    y_offset=104,
    x_min=0,
    x_max=25,
    y_min=75,
    y_max=120
) 

In [4]:
# Instantiate the RSG ecosystem with the connected Gilson and AL1000
rsg = RSG(gilson=g, pump=syr_pump, syringe_diameter=4.606)

In [5]:
# Instantiate the segmentation ecosystem with the dim, runze valve, knauer pump and RSG connected
seg = Segmentation(dim=dim, runze=runze, carrier_pump=k_pump, rsg=rsg)

[Pump] Flow stopped
[Runze] Valve at pos 1 -> state = GAS_TO_DIM
[Runze] valve moved to pos 2 -> state = CARRIER_TO_DIM
[DIM] Valve already at A -> state = DIMState.INJECT


In [21]:
k_pump.set_flow_rate(700)
k_pump.start_flow()

[k_pump] Flow rate set to 700 uL/min
[Pump] Flow started


'OK'

In [22]:
k_pump.stop_flow()

[Pump] Flow stopped


'OK'

In [10]:
seg.prime_gas_path(2)

[Runze] Valve at pos 2 -> state = CARRIER_TO_DIM
[Runze] valve moved to pos 1 -> state = GAS_TO_DIM
[DIM] Valve already at A -> state = DIMState.INJECT
[Segmentation] Phase: IDLE -> GAS_PRIMED


In [19]:
dim.state

<DIMState.INJECT: 2>

In [15]:
runze.carrier_to_dim()

[Runze] Valve at pos 1 -> state = GAS_TO_DIM
[Runze] valve moved to pos 2 -> state = CARRIER_TO_DIM


In [51]:
g.go_into_vial("rack2", 2)

[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127.0


(np.float64(319.0), np.float64(106.0), 20)

In [54]:
rate = 0.5          # mL/min
volume = 50           # uL
direction = "INF"     # WDR = withdraw, INF = Infuse

syr_pump.prepare(rate=rate, volume=volume, direction=direction)

Sent: @00*PHN1
Sent: @00*RAT C 0.5 MM
Sent: @00*VOL50
Sent: @00*DIRINF


In [55]:
syr_pump.start()

Sent: @00*RUN 1


'00I'

In [20]:
k_pump.set_flow_rate(1000)

k_pump.start_flow()

[k_pump] Flow rate set to 1000 uL/min
[Pump] Flow started


'OK'

In [10]:
g.go_into_dim()

[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127


(9.0, 104.0, 79)

In [6]:
seg.prime_with_solvent(flowrate_ul_min = 1000, duration_s = 20)

[Runze] Valve already at pos 2 -> state = CARRIER_TO_DIM
[DIM] Valve already at A -> state = DIMState.INJECT
[k_pump] Flow rate set to 1000 uL/min
[Pump] Flow started
[Pump] Flow stopped


In [56]:
k_pump.stop_flow()

[Pump] Flow stopped


'OK'

In [14]:
g.go_into_vial("rack2", 2)

[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127


(np.float64(319.0), np.float64(106.0), 20)

In [36]:
rsg.take_air_gap(100)

Sent: @00*PHN1
Sent: @00*RAT C 0.05 MM
Sent: @00*VOL100
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP


In [38]:
time.sleep(3)
runze.gas_to_dim()

[Runze] Valve at pos 2 -> state = CARRIER_TO_DIM
[Runze] valve moved to pos 1 -> state = GAS_TO_DIM


In [40]:
rsg.pickup_from_vial("rack1", 1, 50)

[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127.0
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127
Sent: @00*PHN1
Sent: @00*RAT C 0.05 MM
Sent: @00*VOL50
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP


In [43]:
dim.load()

[DIM] Valve moved to B -> state = DIMState.LOAD


In [44]:
rsg.dispense_in_dim(75)

[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127.0
Sent: @00*PHN1
Sent: @00*RAT C 0.5 MM
Sent: @00*VOL75
Sent: @00*DIRINF
Sent: @00*RUN 1
Sent: @00*STP


In [46]:
rate = 0.5          # mL/min
volume = 15           # uL
direction = "INF"     # WDR = withdraw, INF = Infuse

syr_pump.prepare(rate=rate, volume=volume, direction=direction)
syr_pump.start()

Sent: @00*PHN1
Sent: @00*RAT C 0.5 MM
Sent: @00*VOL15
Sent: @00*DIRINF
Sent: @00*RUN 1


'00I'

In [47]:
syr_pump.start()

Sent: @00*RUN 1


'00I'

In [49]:
runze.gas_to_dim()

[Runze] Valve at pos 2 -> state = CARRIER_TO_DIM
[Runze] valve moved to pos 1 -> state = GAS_TO_DIM


In [50]:
dim.inject()
runze.carrier_to_dim()

[DIM] Valve moved to A -> state = DIMState.INJECT
[Runze] Valve at pos 1 -> state = GAS_TO_DIM
[Runze] valve moved to pos 2 -> state = CARRIER_TO_DIM


In [42]:
print(dim.state)
print(runze.state)

DIMState.INJECT
RunzeState.GAS_TO_DIM


In [39]:
reaction_plan = [
    {
        "module": "rack1",   # which module the vials in
        "vial": 1,   #which vial
        "volume": 50   #uL to withdraw
    }
]

result = seg.prepare_slug(reaction_plan)

RuntimeError: Invalid phase transition: required GAS_PRIMED, but current phase is LOOP_LOADED

In [23]:
# Note - the above "prepare_slug" method does not deliver the dyed slug to the sample loop when
# commanded to infuse the same volume it lifted. Use this to ensure the whole slug reaches the loop

rate = 0.5          # mL/min
volume = 15           # uL
direction = "INF"     # WDR = withdraw, INF = Infuse

syr_pump.prepare(rate=rate, volume=volume, direction=direction)

Sent: @00*PHN1
Sent: @00*RAT C 0.5 MM
Sent: @00*VOL15
Sent: @00*DIRINF


In [24]:
syr_pump.start()

Sent: @00*RUN 1


'00I'

In [25]:
time.sleep(10)
g.go_to_vial("rack2", 2)

[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127.0


(np.float64(319.0), np.float64(106.0))

In [28]:
g.go_to_vial("rack2", 2)

[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127.0


(np.float64(319.0), np.float64(106.0))

In [26]:
dim.state

<DIMState.LOAD: 1>